Main Code

In [76]:
import numpy as np
import pandas as pd
import random
from collections import defaultdict
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# Load the dataset
df = pd.read_csv("Day2Data.csv")

# List of features to use for learning (including CINR)
features = ['PCellRSRP', 'PCellRSRQ', 'NeighCellRSRP', 'NeighCellRSRQ', 'CINR']
target = 'Handover'  # The target variable for logistic regression

# Ensure the dataset only contains numeric columns (features + target)
df = df[features + [target]]

# Standardize the features (important for Logistic Regression)
scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])

# Train Logistic Regression Model for the 'Handover' prediction
X = df[features].values  # Features
y = df[target].values    # Target variable (Handover)

logreg_model = LogisticRegression()
logreg_model.fit(X, y)  # Train the model

# Initialize Q-table (state-action value table)
q_table = defaultdict(lambda: [0, 0])  # Q[state][action] for two actions: 0 = No Handover, 1 = Handover

# Hyperparameters for Q-learning
alpha = 0.1  # Learning rate
gamma = 0.9  # Discount factor
epsilon = 4  # Initial exploration rate (epsilon-greedy)
epsilon_min = 0.01  # Minimum epsilon value
epsilon_decay = 0.995  # Decay factor for epsilon

# Count of handovers
handover_count = 0

# Reward function with additional conditions
def reward_function(action, state):
    global handover_count
    
    # Ensure that state values are numerical before proceeding
    state = np.array(state, dtype=np.float64)
    
    # Extract relevant values from the state (normalized features)
    pcell_rsrp = state[0]
    pcell_rsrq = state[1]
    neighcell_rsrp = state[2]
    neighcell_rsrq = state[3]
    cinr = state[4]
    
    if action == 1:  # Handover
        # Increment handover count
        handover_count += 1
        
        # Modify reward function to increase handovers
        if neighcell_rsrp <= pcell_rsrp or neighcell_rsrq <= pcell_rsrq:
            reward = -3  # Smaller penalty for poor handover (allowing more handovers)
        else:
            reward = -0.5  # Moderate penalty for handover if conditions are better but not optimal
        
        # Additional penalties based on CINR value after handover
        if cinr < 5:  # Low CINR value after handover (poor quality)
            reward -= 1  # Extra penalty for bad quality after handover
        elif cinr > 20:  # High CINR value after handover (good quality)
            reward += 3  # Larger reward for good quality after handover

        # Additional reward based on RSRP and RSRQ values
        if neighcell_rsrp > pcell_rsrp and neighcell_rsrq > pcell_rsrq:
            reward += 4  # Larger reward for choosing a better neighboring cell
        
    else:  # No Handover
        # Positive reward for no handover (encourage maintaining the current connection)
        if cinr > 15:  # Good CINR value without handover
            reward = 3  # Reward for maintaining good quality without handover
        else:
            reward = 1  # Mild reward for staying connected with moderate quality
            
    return reward

# Q-learning loop
episodes = 1000  # Number of episodes to train
handover_counts = []  # To track the number of handovers during each episode
for episode in range(episodes):
    # Select a random sample from the dataset
    sample = df.sample(1).values[0]  # Get a single random row as an array
    state = sample[:-1]  # Exclude the last column 'Handover' from state (state is now a list of 5 values)
    
    # Choose action using epsilon-greedy strategy with exploration rate decay
    if random.uniform(0,2) < epsilon:
        # Exploration: Use the Logistic Regression model to predict the probability of a handover
        handover_probability = logreg_model.predict_proba([state])[0][1]  # Probability for handover (class 1)
        
        # Choose action based on the predicted probability
        if handover_probability > 0.3:  # If probability of handover is > 50%, choose handover
            action = 1  # Handover
        else:
            action = 0  # No Handover
    else:
        # Exploitation: Use Q-learning decision based on the Q-table
        action = np.argmax(q_table[tuple(state)])  # Best action (exploit Q-values)

    # Take the chosen action and get the reward
    reward = reward_function(action, state)

    # Get the next state (for simplicity, we assume it stays the same)
    next_state = state  # In practice, you can have a dynamic transition, but we keep it simple here

    # Update Q-value using the Q-learning formula
    best_next_action = np.argmax(q_table[tuple(next_state)])
    q_table[tuple(state)][action] = q_table[tuple(state)][action] + alpha * (reward + gamma * q_table[tuple(next_state)][best_next_action] - q_table[tuple(state)][action])

    # Decay the exploration rate
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay  # Gradually decay epsilon towards the minimum

    # Log handover count after each episode
    handover_counts.append(handover_count)

# After training, print the total number of handovers during training
print(f"Number of handovers during training: {handover_count}")


# ---------------------------------------------------------------
# Evaluation Phase (Using Q-table without updating it)
# ---------------------------------------------------------------

# This function only uses the Q-table for decision making
def evaluate_handovers(df, epsilon):
    global evaluation_handover_count

    # Initialize count of handovers for evaluation
    evaluation_handover_count = 0

    # Iterate through the dataset
    for index, row in df.iterrows():
        state = row[features].values  # State as features (normalized)

        # Use Q-learning decision based on the pre-trained Q-table (no updates)
        action = np.argmax(q_table[tuple(state)])  # Best action (exploit Q-values)

        # Increment the handover count if a handover is performed
        if action == 1:
            evaluation_handover_count += 1

    return evaluation_handover_count

# Set epsilon to the desired value (can be adjusted as needed)
epsilon_value = epsilon  # For example, use the final epsilon from training or set a value here
# Evaluate optimized handovers using the pre-trained Q-table
optimized_handover_count = evaluate_handovers(df, epsilon_value)
# Print the total number of optimized handovers performed during evaluation
print(f"Total optimized handovers performed during evaluation: {optimized_handover_count}")

# Create a dataframe to store results with handover evaluation
def create_handover_evaluation_df(df, epsilon):
    # Initialize a list to store the evaluation results
    evaluation_results = []

    # Iterate through the dataset and evaluate handovers
    for index, row in df.iterrows():
        state = row[features].values  # State as features (normalized)

        # Use Q-learning decision based on the pre-trained Q-table (no updates)
        action = np.argmax(q_table[tuple(state)])  # Best action (exploit Q-values)

        # Append the evaluation result for this sample (using index or row number)
        evaluation_results.append({
            'Index': index,  # Using the index of the row as the identifier
            'Handover': action
        })

    # Create a dataframe from the evaluation results
    evaluation_df = pd.DataFrame(evaluation_results)

    return evaluation_df

# Create the evaluation dataframe after handover evaluation
evaluation_df = create_handover_evaluation_df(df, epsilon_value)
print(evaluation_df['Handover'].value_counts())


Number of handovers during training: 97
Total optimized handovers performed during evaluation: 41
Handover
0    606
1     41
Name: count, dtype: int64


In [67]:
import numpy as np
import pandas as pd
import random
from collections import defaultdict
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Load the dataset
df = pd.read_csv("Day2Data.csv")

# List of features to use for learning (including CINR)
features = ['PCellRSRP', 'PCellRSRQ', 'NeighCellRSRP', 'NeighCellRSRQ', 'CINR']
target = 'Handover'  # The target variable for logistic regression

# Ensure the dataset only contains numeric columns (features + target)
df = df[features + [target]]

# Standardize the features (important for Logistic Regression)
scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])

# Train Logistic Regression Model for the 'Handover' prediction
X = df[features].values  # Features
y = df[target].values    # Target variable (Handover)

logreg_model = LogisticRegression()
logreg_model.fit(X, y)  # Train the model

# Initialize Q-table (state-action value table)
q_table = defaultdict(lambda: [0, 0])  # Q[state][action] for two actions: 0 = No Handover, 1 = Handover

# Hyperparameters for Q-learning
alpha = 0.1  # Learning rate
gamma = 0.9  # Discount factor
epsilon = 2  # Initial exploration rate (epsilon-greedy)
epsilon_min = 0.01  # Minimum epsilon value
epsilon_decay = 0.995  # Decay factor for epsilon

# Count of handovers during training
handover_count = 0

# Reward function with additional conditions
def reward_function(action, state):
    global handover_count
    
    # Ensure that state values are numerical before proceeding
    state = np.array(state, dtype=np.float64)
    
    # Extract relevant values from the state (normalized features)
    pcell_rsrp = state[0]
    pcell_rsrq = state[1]
    neighcell_rsrp = state[2]
    neighcell_rsrq = state[3]
    cinr = state[4]
    
    if action == 1:  # Handover
        # Increment handover count
        handover_count += 1
        
        # Penalize if the neighbor cell values are not better than the serving cell values
        if neighcell_rsrp <= pcell_rsrp or neighcell_rsrq <= pcell_rsrq:
            reward = -5  # Strong penalty for poor handover
        else:
            reward = -1  # Moderate penalty for handover if conditions are better but not optimal
        
        # Additional penalties based on CINR value after handover
        if cinr < 5:  # Low CINR value after handover (poor quality)
            reward -= 3  # Extra penalty for bad quality after handover
        elif cinr > 20:  # High CINR value after handover (good quality)
            reward += 1  # Small reward for good quality after handover

        # Additional reward based on RSRP and RSRQ values
        if neighcell_rsrp > pcell_rsrp and neighcell_rsrq > pcell_rsrq:
            reward += 2  # Extra reward for choosing a better neighboring cell
        
    else:  # No Handover
        # Positive reward for no handover (encourage maintaining the current connection)
        if cinr > 15:  # Good CINR value without handover
            reward = 2  # Reward for maintaining good quality without handover
        else:
            reward = 1  # Mild reward for staying connected with moderate quality
            
    return reward

# Q-learning loop (training phase)
episodes = 1000  # Number of episodes to train
for episode in range(episodes):
    # Select a random sample from the dataset
    sample = df.sample(1).values[0]  # Get a single random row as an array
    state = sample[:-1]  # Exclude the last column 'Handover' from state (state is now a list of 5 values)
    
    # Choose action using epsilon-greedy strategy with exploration rate decay
    if random.uniform(0, 1) < epsilon:
        # Exploration: Use the Logistic Regression model to predict the probability of a handover
        handover_probability = logreg_model.predict_proba([state])[0][1]  # Probability for handover (class 1)
        
        # Choose action based on the predicted probability
        if handover_probability > 0.3:  # If probability of handover is > 30%, choose handover
            action = 1  # Handover
        else:
            action = 0  # No Handover
    else:
        # Exploitation: Use Q-learning decision based on the Q-table
        action = np.argmax(q_table[tuple(state)])  # Best action (exploit Q-values)

    # Take the chosen action and get the reward
    reward = reward_function(action, state)

    # Get the next state (for simplicity, we assume it stays the same)
    next_state = state  # In practice, you can have a dynamic transition, but we keep it simple here

    # Update Q-value using the Q-learning formula
    best_next_action = np.argmax(q_table[tuple(next_state)])
    q_table[tuple(state)][action] = q_table[tuple(state)][action] + alpha * (reward + gamma * q_table[tuple(next_state)][best_next_action] - q_table[tuple(state)][action])

    # Decay the exploration rate
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay  # Gradually decay epsilon towards the minimum


# Print the total number of handovers performed during training
print(f"Total handovers performed: {handover_count}")


Total handovers performed: 63


In [70]:
# ---------------------------------------------------------------
# Evaluation Phase (Using trained Q-table without updating it)
# ---------------------------------------------------------------

# This function takes epsilon as an input parameter, allowing it to be controlled by the user
def evaluate_handovers(df, epsilon):
    global evaluation_handover_count

    # Initialize count of handovers for evaluation
    evaluation_handover_count = 0

    # Iterate through the dataset
    for index, row in df.iterrows():
        state = row[features].values  # State as features (normalized)

        # Choose action using epsilon-greedy strategy with the trained Q-table (no updates)
        if random.uniform(0, 1) < epsilon:
            # Exploration: Choose action using the Logistic Regression model
            handover_probability = logreg_model.predict_proba([state])[0][1]  # Probability for handover (class 1)

            # Choose action based on the predicted probability
            if handover_probability > 0.5:  # If probability of handover is > 30%, choose handover
                action = 1  # Handover
            else:
                action = 0  # No Handover
        else:
            # Exploitation: Use Q-learning decision based on the pre-trained Q-table
            action = np.argmax(q_table[tuple(state)])  # Best action (exploit Q-values)

        # Evaluate reward based on the Q-table (no updates to Q-table)
        reward = reward_function(action, state)

        # Increment the handover count if a handover is performed
        if action == 1:
            evaluation_handover_count += 1

    return evaluation_handover_count


# Set epsilon to the desired value (can be adjusted as needed)
# The same epsilon value used during training will be used for evaluation
epsilon_value = epsilon  # For example, use the final epsilon from training or set a value here

# Evaluate optimized handovers using the pre-trained Q-table
optimized_handover_count = evaluate_handovers(df, epsilon_value)

# Print the total number of optimized handovers performed during evaluation
print(f"Total optimized handovers performed during evaluation: {optimized_handover_count}")


Total optimized handovers performed during evaluation: 1


In [75]:
import numpy as np
import pandas as pd
import random
from collections import defaultdict
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Load the dataset
df = pd.read_csv("Day2Data.csv")

# List of features to use for learning (including CINR)
features = ['PCellRSRP', 'PCellRSRQ', 'NeighCellRSRP', 'NeighCellRSRQ', 'CINR']
target = 'Handover'  # The target variable for logistic regression

# Ensure the dataset only contains numeric columns (features + target)
df = df[features + [target]]

# Standardize the features (important for Logistic Regression)
scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])

# Train Logistic Regression Model for the 'Handover' prediction
X = df[features].values  # Features
y = df[target].values    # Target variable (Handover)

logreg_model = LogisticRegression()
logreg_model.fit(X, y)  # Train the model

# Initialize Q-table (state-action value table)
q_table = defaultdict(lambda: [0, 0])  # Q[state][action] for two actions: 0 = No Handover, 1 = Handover

# Hyperparameters for Q-learning
alpha = 0.1  # Learning rate
gamma = 0.9  # Discount factor
epsilon = 4  # Initial exploration rate (epsilon-greedy)
epsilon_min = 0.01  # Minimum epsilon value
epsilon_decay = 0.995  # Decay factor for epsilon

# Count of handovers
handover_count = 0

# Reward function with additional conditions
def reward_function(action, state):
    global handover_count
    
    # Ensure that state values are numerical before proceeding
    state = np.array(state, dtype=np.float64)
    
    # Extract relevant values from the state (normalized features)
    pcell_rsrp = state[0]
    pcell_rsrq = state[1]
    neighcell_rsrp = state[2]
    neighcell_rsrq = state[3]
    cinr = state[4]
    
    if action == 1:  # Handover
        # Increment handover count
        handover_count += 1
        
        # Modify reward function to increase handovers
        if neighcell_rsrp <= pcell_rsrp or neighcell_rsrq <= pcell_rsrq:
            reward = -3  # Smaller penalty for poor handover (allowing more handovers)
        else:
            reward = -0.5  # Moderate penalty for handover if conditions are better but not optimal
        
        # Additional penalties based on CINR value after handover
        if cinr < 5:  # Low CINR value after handover (poor quality)
            reward -= 1  # Extra penalty for bad quality after handover
        elif cinr > 20:  # High CINR value after handover (good quality)
            reward += 3  # Larger reward for good quality after handover

        # Additional reward based on RSRP and RSRQ values
        if neighcell_rsrp > pcell_rsrp and neighcell_rsrq > pcell_rsrq:
            reward += 4  # Larger reward for choosing a better neighboring cell
        
    else:  # No Handover
        # Positive reward for no handover (encourage maintaining the current connection)
        if cinr > 15:  # Good CINR value without handover
            reward = 3  # Reward for maintaining good quality without handover
        else:
            reward = 1  # Mild reward for staying connected with moderate quality
            
    return reward

# Q-learning loop
episodes = 1000  # Number of episodes to train
for episode in range(episodes):
    # Select a random sample from the dataset
    sample = df.sample(1).values[0]  # Get a single random row as an array
    state = sample[:-1]  # Exclude the last column 'Handover' from state (state is now a list of 5 values)
    
    # Choose action using epsilon-greedy strategy with exploration rate decay
    if random.uniform(0, 1) < epsilon:
        # Exploration: Use the Logistic Regression model to predict the probability of a handover
        handover_probability = logreg_model.predict_proba([state])[0][1]  # Probability for handover (class 1)
        
        # Choose action based on the predicted probability
        if handover_probability > 0.3:  # If probability of handover is > 50%, choose handover
            action = 1  # Handover
        else:
            action = 0  # No Handover
    else:
        # Exploitation: Use Q-learning decision based on the Q-table
        action = np.argmax(q_table[tuple(state)])  # Best action (exploit Q-values)

    # Take the chosen action and get the reward
    reward = reward_function(action, state)

    # Get the next state (for simplicity, we assume it stays the same)
    next_state = state  # In practice, you can have a dynamic transition, but we keep it simple here

    # Update Q-value using the Q-learning formula
    best_next_action = np.argmax(q_table[tuple(next_state)])
    q_table[tuple(state)][action] = q_table[tuple(state)][action] + alpha * (reward + gamma * q_table[tuple(next_state)][best_next_action] - q_table[tuple(state)][action])

    # Decay the exploration rate
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay  # Gradually decay epsilon towards the minimum
# Assuming q_table is already trained (from the previous code snippet)
import pandas as pd

# Convert the Q-table (defaultdict) into a pandas DataFrame
q_table_data = []
for state, actions in q_table.items():
    # Convert the state tuple and actions (Q-values) into a row for the DataFrame
    state_values = list(state)
    q_table_data.append(state_values + actions)

# Create DataFrame
q_table_df = pd.DataFrame(q_table_data, columns=features + ['Q0', 'Q1'])

# Save to CSV
q_table_df.to_csv("q_table.csv", index=False)

print("Q-table has been saved to q_table.csv")

# ---------------------------------------------------------------
# Evaluation Phase (Using trained Q-table without updating it)
# ---------------------------------------------------------------

# This function takes epsilon as an input parameter, allowing it to be controlled by the user
def evaluate_handovers(df, epsilon):
    global evaluation_handover_count

    # Initialize count of handovers for evaluation
    evaluation_handover_count = 0

    # Iterate through the dataset
    for index, row in df.iterrows():
        state = row[features].values  # State as features (normalized)

        # Choose action using epsilon-greedy strategy with the trained Q-table (no updates)
        if random.uniform(0, 1) < epsilon:
            # Exploration: Choose action using the Logistic Regression model
            handover_probability = logreg_model.predict_proba([state])[0][1]  # Probability for handover (class 1)

            # Choose action based on the predicted probability
            if handover_probability > 0.3:  # If probability of handover is > 30%, choose handover
                action = 1  # Handover
            else:
                action = 0  # No Handover
        else:
            # Exploitation: Use Q-learning decision based on the pre-trained Q-table
            action = np.argmax(q_table[tuple(state)])  # Best action (exploit Q-values)

        # Evaluate reward based on the Q-table (no updates to Q-table)
        reward = reward_function(action, state)

        # Increment the handover count if a handover is performed
        if action == 1:
            evaluation_handover_count += 1

    return evaluation_handover_count


# Set epsilon to the desired value (can be adjusted as needed)
# The same epsilon value used during training will be used for evaluation
epsilon_value = epsilon  # For example, use the final epsilon from training or set a value here

# Evaluate optimized handovers using the pre-trained Q-table
optimized_handover_count = evaluate_handovers(df, epsilon_value)

# Print the total number of optimized handovers performed during evaluation
print(f"Total optimized handovers performed during evaluation: {optimized_handover_count}")


Q-table has been saved to q_table.csv
Total optimized handovers performed during evaluation: 54


In [74]:
import numpy as np
import pandas as pd
import random
from collections import defaultdict
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# Load the dataset
df = pd.read_csv("Day2Data.csv")

# List of features to use for learning (including CINR)
features = ['PCellRSRP', 'PCellRSRQ', 'NeighCellRSRP', 'NeighCellRSRQ', 'CINR']
target = 'Handover'  # The target variable for logistic regression

# Ensure the dataset only contains numeric columns (features + target)
df = df[features + [target]]

# Standardize the features (important for Logistic Regression)
scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])

# Train Logistic Regression Model for the 'Handover' prediction
X = df[features].values  # Features
y = df[target].values    # Target variable (Handover)

logreg_model = LogisticRegression()
logreg_model.fit(X, y)  # Train the model

# Initialize Q-table (state-action value table)
q_table = defaultdict(lambda: [0, 0])  # Q[state][action] for two actions: 0 = No Handover, 1 = Handover

# Hyperparameters for Q-learning
alpha = 0.1  # Learning rate
gamma = 0.9  # Discount factor
epsilon = 2  # Initial exploration rate (epsilon-greedy)
epsilon_min = 0.01  # Minimum epsilon value
epsilon_decay = 0.995  # Decay factor for epsilon

# Count of handovers
handover_count = 0

# Reward function with additional conditions
def reward_function(action, state):
    global handover_count
    
    # Ensure that state values are numerical before proceeding
    state = np.array(state, dtype=np.float64)
    
    # Extract relevant values from the state (normalized features)
    pcell_rsrp = state[0]
    pcell_rsrq = state[1]
    neighcell_rsrp = state[2]
    neighcell_rsrq = state[3]
    cinr = state[4]
    
    if action == 1:  # Handover
        # Increment handover count
        handover_count += 1
        
        # Modify reward function to increase handovers
        if neighcell_rsrp <= pcell_rsrp or neighcell_rsrq <= pcell_rsrq:
            reward = -3  # Smaller penalty for poor handover (allowing more handovers)
        else:
            reward = -0.5  # Moderate penalty for handover if conditions are better but not optimal
        
        # Additional penalties based on CINR value after handover
        if cinr < 5:  # Low CINR value after handover (poor quality)
            reward -= 1  # Extra penalty for bad quality after handover
        elif cinr > 20:  # High CINR value after handover (good quality)
            reward += 3  # Larger reward for good quality after handover

        # Additional reward based on RSRP and RSRQ values
        if neighcell_rsrp > pcell_rsrp and neighcell_rsrq > pcell_rsrq:
            reward += 4  # Larger reward for choosing a better neighboring cell
        
    else:  # No Handover
        # Positive reward for no handover (encourage maintaining the current connection)
        if cinr > 15:  # Good CINR value without handover
            reward = 3  # Reward for maintaining good quality without handover
        else:
            reward = 1  # Mild reward for staying connected with moderate quality
            
    return reward

# Q-learning loop
episodes = 1000  # Number of episodes to train
handover_counts = []  # To track the number of handovers during each episode
for episode in range(episodes):
    # Select a random sample from the dataset
    sample = df.sample(1).values[0]  # Get a single random row as an array
    state = sample[:-1]  # Exclude the last column 'Handover' from state (state is now a list of 5 values)
    
    # Choose action using epsilon-greedy strategy with exploration rate decay
    if random.uniform(0, 1) < epsilon:
        # Exploration: Use the Logistic Regression model to predict the probability of a handover
        handover_probability = logreg_model.predict_proba([state])[0][1]  # Probability for handover (class 1)
        
        # Choose action based on the predicted probability
        if handover_probability > 0.3:  # If probability of handover is > 50%, choose handover
            action = 1  # Handover
        else:
            action = 0  # No Handover
    else:
        # Exploitation: Use Q-learning decision based on the Q-table
        action = np.argmax(q_table[tuple(state)])  # Best action (exploit Q-values)

    # Take the chosen action and get the reward
    reward = reward_function(action, state)

    # Get the next state (for simplicity, we assume it stays the same)
    next_state = state  # In practice, you can have a dynamic transition, but we keep it simple here

    # Update Q-value using the Q-learning formula
    best_next_action = np.argmax(q_table[tuple(next_state)])
    q_table[tuple(state)][action] = q_table[tuple(state)][action] + alpha * (reward + gamma * q_table[tuple(next_state)][best_next_action] - q_table[tuple(state)][action])

    # Decay the exploration rate
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay  # Gradually decay epsilon towards the minimum

    # Log handover count after each episode
    handover_counts.append(handover_count)

# After training, print the total number of handovers during training
print(f"Number of handovers during training: {handover_count}")


# ---------------------------------------------------------------
# Evaluation Phase (Using trained Q-table without updating it)
# ---------------------------------------------------------------

# This function takes epsilon as an input parameter, allowing it to be controlled by the user
def evaluate_handovers(df, epsilon):
    global evaluation_handover_count

    # Initialize count of handovers for evaluation
    evaluation_handover_count = 0

    # Iterate through the dataset
    for index, row in df.iterrows():
        state = row[features].values  # State as features (normalized)

        # Choose action using epsilon-greedy strategy with the trained Q-table (no updates)
        if random.uniform (0,1) < epsilon:
            # Exploration: Choose action using the Logistic Regression model
            handover_probability = logreg_model.predict_proba([state])[0][1]  # Probability for handover (class 1)

            # Choose action based on the predicted probability
            if handover_probability > 0.3:  # If probability of handover is > 30%, choose handover
                action = 1  # Handover
            else:
                action = 0  # No Handover
        else:
            # Exploitation: Use Q-learning decision based on the pre-trained Q-table
            action = np.argmax(q_table[tuple(state)])  # Best action (exploit Q-values)

        # Evaluate reward based on the Q-table (no updates to Q-table)
        reward = reward_function(action, state)

        # Increment the handover count if a handover is performed
        if action == 1:
            evaluation_handover_count += 1

    return evaluation_handover_count

# Set epsilon to the desired value (can be adjusted as needed)
epsilon_value = epsilon  # For example, use the final epsilon from training or set a value here
# Evaluate optimized handovers using the pre-trained Q-table
optimized_handover_count = evaluate_handovers(df, epsilon_value)
# Print the total number of optimized handovers performed during evaluation
print(f"Total optimized handovers performed during evaluation: {optimized_handover_count}")

# Create a dataframe to store results with handover evaluation
def create_handover_evaluation_df(df, epsilon):
    # Initialize a list to store the evaluation results
    evaluation_results = []

    # Iterate through the dataset and evaluate handovers
    for index, row in df.iterrows():
        state = row[features].values  # State as features (normalized)

        # Choose action using epsilon-greedy strategy with the trained Q-table (no updates)
        if random.uniform(0,1) < epsilon:
            # Exploration: Choose action using the Logistic Regression model
            handover_probability = logreg_model.predict_proba([state])[0][1]  # Probability for handover (class 1)
            
            # Choose action based on the predicted probability
            if handover_probability > 0.3:  # If probability of handover is > 30%, choose handover
                action = 1  # Handover
            else:
                action = 0  # No Handover
        else:
            # Exploitation: Use Q-learning decision based on the pre-trained Q-table
            action = np.argmax(q_table[tuple(state)])  # Best action (exploit Q-values)

        # Append the evaluation result for this sample (using index or row number)
        evaluation_results.append({
            'Index': index,  # Using the index of the row as the identifier
            'Handover': action
        })

    # Create a dataframe from the evaluation results
    evaluation_df = pd.DataFrame(evaluation_results)

    return evaluation_df

# Create the evaluation dataframe after handover evaluation
evaluation_df = create_handover_evaluation_df(df, epsilon_value)
evaluation_df['Handover'].value_counts()


Number of handovers during training: 92
Total optimized handovers performed during evaluation: 44


Handover
0    603
1     44
Name: count, dtype: int64

In [30]:

# Create the evaluation dataframe after handover evaluation
evaluation_df = create_handover_evaluation_df(df, epsilon_value)

# Save the evaluation results to a CSV file
evaluation_df.to_csv("handover_evaluation_results2.csv", index=False)

# Display a confirmation message
print("Evaluation results saved to 'handover_evaluation_results.csv'")


Evaluation results saved to 'handover_evaluation_results.csv'


In [26]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Load the dataset
df = pd.read_csv("Day2Data.csv")

# Define features and target
features = ['PCellRSRP', 'PCellRSRQ', 'NeighCellRSRP', 'NeighCellRSRQ', 'CINR']
target = 'Handover'

# Filter required columns
df = df[features + [target]]

# Normalize features
scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])

# Train Logistic Regression model
X = df[features].values
y = df[target].values

logreg_model = LogisticRegression()
logreg_model.fit(X, y)

# Predict handovers across the dataset
predictions = logreg_model.predict(X)
df['Predicted_Handover'] = predictions

# Count total handover predictions
total_handovers = np.sum(predictions)

# Display results
print(f"Total predicted handovers: {total_handovers}")


Total predicted handovers: 52
